# 02 · Silver → Gold

Builds the **reporting star schema** that Power BI, the Streamlit dashboard and the AI analyst read, then runs
production tests against it. Any `error`-severity test failure stops the pipeline, so a broken build is never
published over the live warehouse.

| Table | Grain | Notes |
|---|---|---|
| `dim_date` | day | covers every fact, budget and AP due date; fiscal helpers |
| `dim_department` / `dim_department_history` | department / version | type 1 current view + SCD2 history; `-1` unknown member |
| `dim_gl_account` | GL account | normalised category, parent account, `-1` unknown |
| `dim_vendor` | vendor | survivors only + **inferred members** for unregistered suppliers; `-1` unknown, `-2` n/a |
| `dim_customer` | customer | canonical customers; `-1` unknown, `-2` n/a |
| `fact_transactions` | posted transaction | ~2M rows, test customers excluded, USD + local amounts |
| `fact_budget` | version × month × dept × account | `is_latest_version` picks the newest reforecast |
| `fact_ap_invoices` | invoice | open amount, days past due, aging bucket |
| `kpi_monthly_actual_vs_budget`, `monthly_financials`, `kpi_ap_aging`, `kpi_revenue_by_segment` | aggregates | dashboard-ready |

In [1]:
DB_PATH = "../data/warehouse.duckdb"
RUN_ID = None
AS_OF_DATE = None
MEMORY_LIMIT = "1500MB"

In [2]:
import time
import uuid
from datetime import date, datetime, timezone
from pathlib import Path

import duckdb

RUN_ID = RUN_ID or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
AS_OF = date.fromisoformat(AS_OF_DATE) if AS_OF_DATE else date.today()

con = duckdb.connect(str(Path(DB_PATH).resolve()))
con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET TimeZone = 'UTC'")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

try:
    display
except NameError:
    display = print


def show(sql: str, limit: int = 50):
    df = con.sql(sql).limit(limit).df()
    display(df)
    return df


def scalar(sql: str):
    return con.execute(sql).fetchone()[0]


_t0 = time.time()
print(f"run_id={RUN_ID}  as_of={AS_OF}")

run_id=20261023T201356Z-0cba4d  as_of=2026-10-24


## 1 · Dimensions
### dim_date

In [3]:
con.execute(f"""
CREATE OR REPLACE TABLE gold.dim_date AS
WITH bounds AS (
    SELECT least((SELECT min(event_date) FROM silver.silver_transactions),
                 (SELECT min(month_start) FROM silver.silver_budgets),
                 (SELECT min(invoice_date) FROM silver.ap_invoices)) AS d0,
           greatest((SELECT max(event_date) FROM silver.silver_transactions),
                    (SELECT last_day(max(month_start)) FROM silver.silver_budgets),
                    (SELECT max(due_date) FROM silver.ap_invoices),
                    DATE '{AS_OF}') AS d1
), days AS (
    SELECT unnest(generate_series(d0, d1, INTERVAL 1 DAY))::DATE AS d FROM bounds
)
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_sk,
       d AS full_date,
       year(d) AS year_no,
       quarter(d) AS quarter_no,
       month(d) AS month_no,
       strftime(d, '%B') AS month_name,
       strftime(d, '%Y-%m') AS year_month,
       weekofyear(d) AS week_no,
       dayofweek(d) AS day_of_week,
       dayofweek(d) IN (0, 6) AS is_weekend,
       d = last_day(d) AS is_month_end,
       date_trunc('month', d)::DATE AS month_start,
       'Q' || quarter(d) || ' ' || year(d) AS quarter_label,
       strftime(d, '%a') AS day_name,
       dayofweek(d) NOT IN (0, 6) AS is_business_day,
       d <= DATE '{AS_OF}' AS is_past
FROM days
""")
show("SELECT min(full_date) AS first_day, max(full_date) AS last_day, count(*) AS days FROM gold.dim_date")

first_day,last_day,days
2024-01-01,2026-12-31,1096


### dim_department (current) and dim_department_history (SCD2)
Surrogate keys are assigned in a stable order; `-1` absorbs facts whose department code is not in the master.

In [4]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_department AS
SELECT row_number() OVER (ORDER BY dept_code)::BIGINT AS department_id,
       dept_code, dept_name AS department_name, cost_center, headcount::INTEGER AS headcount,
       region, business_function, is_active
FROM silver.departments
UNION ALL
SELECT -1, 'UNK', 'Unknown department', 'CC-NA', 0, 'Unknown', 'Unknown', false
""")
con.execute("""
CREATE OR REPLACE TABLE gold.dim_department_history AS
SELECT row_number() OVER (ORDER BY h.dept_code, h.valid_from)::BIGINT AS department_version_id,
       d.department_id, h.dept_code, h.dept_name AS department_name, h.cost_center, h.headcount::INTEGER AS headcount,
       h.region, h.is_active, h.valid_from, h.valid_to, h.is_current
FROM silver.departments_scd2 h JOIN gold.dim_department d USING (dept_code)
""")
show("SELECT region, count(*) AS departments, sum(headcount) AS headcount, sum((NOT is_active)::INT) AS inactive "
     "FROM gold.dim_department WHERE department_id > 0 GROUP BY 1")

region,departments,headcount,inactive
NA,21,567.0,0.0
EMEA,20,347.0,1.0


### dim_gl_account

In [5]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_gl_account AS
SELECT row_number() OVER (ORDER BY account_code)::BIGINT AS account_id,
       account_code, account_name, category, parent_account_code,
       CASE WHEN category = 'Revenue' THEN 'Income' WHEN category = 'Other' THEN 'Non-operating' ELSE 'Cost' END AS statement_section,
       is_active
FROM silver.gl_accounts
UNION ALL
SELECT -1, 'UNK', 'Unknown account', 'Unclassified', NULL, 'Cost', false
""")

### dim_vendor — with inferred members
Transactions sometimes name a supplier that procurement never registered. Rather than dropping the spend or lumping
it into "unknown", each distinct unregistered name becomes an **inferred member** (`is_inferred = true`) that a data
steward can later merge into a real vendor.

In [6]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_vendor AS
WITH registered AS (
    SELECT vendor_id AS vendor_code, vendor_name, vendor_category, country, payment_terms_days, is_active, false AS is_inferred
    FROM silver.vendors
), inferred AS (
    SELECT 'INF-' || lpad(row_number() OVER (ORDER BY vendor_name)::VARCHAR, 5, '0') AS vendor_code,
           vendor_name, 'Unregistered' AS vendor_category, 'Unknown' AS country, 30 AS payment_terms_days,
           true AS is_active, true AS is_inferred
    FROM (SELECT DISTINCT vendor_name FROM silver.silver_transactions WHERE vendor_id IS NULL AND vendor_name IS NOT NULL)
)
SELECT row_number() OVER (ORDER BY is_inferred, vendor_code)::BIGINT AS vendor_id, *
FROM (SELECT * FROM registered UNION ALL SELECT * FROM inferred)
UNION ALL SELECT -1, 'UNK', 'Unknown vendor', 'Unclassified', 'Unknown', 30, false, false
UNION ALL SELECT -2, 'N/A', 'Not applicable', 'Not applicable', 'Not applicable', 0, false, false
""")
show("SELECT is_inferred, count(*) AS vendors FROM gold.dim_vendor WHERE vendor_id > 0 GROUP BY 1")

is_inferred,vendors
False,5000
True,405


### dim_customer

In [7]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_customer AS
SELECT row_number() OVER (ORDER BY customer_id)::BIGINT AS customer_id,
       customer_id AS customer_code, customer_name, email, segment, region, country, created_date, is_test
FROM silver.customers
UNION ALL SELECT -1, 'UNK', 'Unknown customer', NULL, 'Unknown', 'Unknown', 'Unknown', NULL, false
UNION ALL SELECT -2, 'N/A', 'Not applicable', NULL, 'Not applicable', 'Not applicable', 'Not applicable', NULL, false
""")
show("SELECT segment, region, count(*) AS customers FROM gold.dim_customer WHERE customer_id > 0 GROUP BY ALL ORDER BY 3 DESC LIMIT 12")

segment,region,customers
SMB,NA,14923
SMB,EMEA,8891
Mid-Market,NA,7391
Mid-Market,EMEA,4609
SMB,APAC,4569
Enterprise,NA,2504
Mid-Market,APAC,2318
SMB,LATAM,1490
Enterprise,EMEA,1432
Enterprise,APAC,766


## 2 · Facts
### fact_transactions
Every silver row maps to exactly one member of every dimension (unknown / not-applicable members instead of NULL
keys), so report joins never lose rows. Transactions of test customers are excluded from reporting.

In [8]:
_t = time.time()
con.execute("""
CREATE OR REPLACE TABLE gold.fact_transactions AS
SELECT row_number() OVER (ORDER BY t.event_ts, t.transaction_id)::BIGINT AS transaction_sk,
       t.transaction_id,
       CAST(strftime(t.event_date, '%Y%m%d') AS INTEGER) AS date_sk,
       coalesce(d.department_id, -1) AS department_id,
       coalesce(a.account_id, -1) AS account_id,
       CASE WHEN t.vendor_name IS NULL THEN -2 ELSE coalesce(v.vendor_id, vi.vendor_id, -1) END AS vendor_id,
       CASE WHEN t.customer_id_raw IS NULL THEN -2 ELSE coalesce(c.customer_id, -1) END AS customer_id,
       t.amount_usd,
       t.currency,
       t.txn_type,
       t.source_channel,
       t.event_ts,
       t.amount_local,
       t.fx_rate,
       t.is_late_arriving,
       t.is_adjustment,
       t.ingested_at
FROM silver.silver_transactions t
LEFT JOIN gold.dim_department d ON d.dept_code = t.dept_code AND d.department_id > 0
LEFT JOIN gold.dim_gl_account a ON a.account_code = t.account_code AND a.account_id > 0
LEFT JOIN gold.dim_vendor v ON v.vendor_code = t.vendor_id AND NOT v.is_inferred
LEFT JOIN gold.dim_vendor vi ON vi.vendor_name = t.vendor_name AND vi.is_inferred AND t.vendor_id IS NULL
LEFT JOIN gold.dim_customer c ON c.customer_code = t.customer_id AND c.customer_id > 0
WHERE NOT t.is_test_customer
""")
print(f"gold.fact_transactions: {scalar('SELECT count(*) FROM gold.fact_transactions'):,} rows in {time.time() - _t:.1f}s")
show("""
SELECT source_channel, count(*) AS rows, round(sum(amount_usd) / 1e6, 1) AS usd_m,
       sum(is_late_arriving::INT) AS late, sum((department_id = -1)::INT) AS unknown_dept,
       sum((vendor_id = -1)::INT) AS unknown_vendor, sum((customer_id = -1)::INT) AS unknown_customer
FROM gold.fact_transactions GROUP BY 1 ORDER BY 2 DESC
""")

gold.fact_transactions: 1,993,263 rows in 10.3s


source_channel,rows,usd_m,late,unknown_dept,unknown_vendor,unknown_customer
erp_backfill,1890936,10384.1,17364.0,3802.0,0.0,283.0
pos_legacy,63396,614.7,557.0,148.0,0.0,27.0
kafka_stream,37431,235.0,315.0,66.0,0.0,6.0
file_upload,1500,11.3,1375.0,0.0,0.0,0.0


### fact_budget
All plan versions are kept for variance-to-plan analysis; `is_latest_version` marks the reforecast that reporting
compares actuals against.

In [9]:
con.execute("""
CREATE OR REPLACE TABLE gold.fact_budget AS
SELECT row_number() OVER (ORDER BY b.month_start, b.dept_code, b.account_code, b.version_rank)::BIGINT AS budget_sk,
       CAST(strftime(b.month_start, '%Y%m%d') AS INTEGER) AS date_sk,
       b.month_start,
       coalesce(d.department_id, -1) AS department_id,
       coalesce(a.account_id, -1) AS account_id,
       b.budget_amount_usd,
       b.budget_version,
       b.version_rank = max(b.version_rank) OVER (PARTITION BY b.month_start, b.dept_code, b.account_code) AS is_latest_version,
       b.submitted_at
FROM silver.silver_budgets b
LEFT JOIN gold.dim_department d ON d.dept_code = b.dept_code AND d.department_id > 0
LEFT JOIN gold.dim_gl_account a ON a.account_code = b.account_code AND a.account_id > 0
""")
show("SELECT budget_version, is_latest_version, count(*) AS lines FROM gold.fact_budget GROUP BY ALL ORDER BY 1, 2")

budget_version,is_latest_version,lines
ORIGINAL,False,17834
ORIGINAL,True,6056
REFORECAST_Q2,False,11849
REFORECAST_Q2,True,6028
REFORECAST_Q3,True,11919


### fact_ap_invoices — aging as of the run date

In [10]:
con.execute(f"""
CREATE OR REPLACE TABLE gold.fact_ap_invoices AS
WITH base AS (
    SELECT i.*,
           CASE WHEN i.status IN ('PAID', 'VOID') THEN 0 ELSE i.amount_usd - i.paid_amount_usd END AS open_amount_usd
    FROM silver.ap_invoices i
)
SELECT row_number() OVER (ORDER BY b.invoice_date, b.invoice_id)::BIGINT AS invoice_sk,
       b.invoice_id, b.invoice_number,
       coalesce(v.vendor_id, -1) AS vendor_id,
       CAST(strftime(b.invoice_date, '%Y%m%d') AS INTEGER) AS invoice_date_sk,
       CAST(strftime(b.due_date, '%Y%m%d') AS INTEGER) AS due_date_sk,
       b.doc_type, b.status, b.currency,
       b.amount_usd, b.paid_amount_usd, b.open_amount_usd::DECIMAL(18, 2) AS open_amount_usd,
       CASE WHEN b.open_amount_usd <> 0 THEN greatest(DATE '{AS_OF}' - b.due_date, 0) END AS days_past_due,
       CASE WHEN b.open_amount_usd = 0 THEN 'Closed'
            WHEN DATE '{AS_OF}' <= b.due_date THEN 'Current'
            WHEN DATE '{AS_OF}' - b.due_date <= 30 THEN '1-30'
            WHEN DATE '{AS_OF}' - b.due_date <= 60 THEN '31-60'
            WHEN DATE '{AS_OF}' - b.due_date <= 90 THEN '61-90'
            ELSE '90+' END AS aging_bucket,
       b.paid_date, b.is_overpaid, b.due_date_repaired
FROM base b
LEFT JOIN gold.dim_vendor v ON v.vendor_code = b.vendor_id AND NOT v.is_inferred
""")
show("SELECT aging_bucket, count(*) AS invoices, round(sum(open_amount_usd) / 1e6, 2) AS open_usd_m "
     "FROM gold.fact_ap_invoices GROUP BY 1 ORDER BY 1")

aging_bucket,invoices,open_usd_m
1-30,1549,8.53
31-60,402,2.36
61-90,140,0.70
90+,200,0.97
Closed,140045,0.00
Current,6146,33.84


## 3 · KPI and aggregate tables
Budget comparisons use the **latest plan version** only, split into revenue targets and expense budgets.
The month in progress is compared month-to-date: its budget is prorated by days elapsed (`is_month_to_date`).

In [11]:
con.execute(f"""
CREATE OR REPLACE TABLE gold.kpi_monthly_actual_vs_budget AS
WITH actuals AS (
    SELECT date_trunc('month', dd.full_date)::DATE AS month_start, f.department_id,
           SUM(CASE WHEN a.category = 'Revenue' THEN f.amount_usd ELSE 0 END) AS revenue_actual,
           SUM(CASE WHEN a.category <> 'Revenue' THEN f.amount_usd ELSE 0 END) AS expense_actual
    FROM gold.fact_transactions f
    JOIN gold.dim_date dd USING (date_sk)
    JOIN gold.dim_gl_account a USING (account_id)
    GROUP BY 1, 2
), budgets AS (
    SELECT b.month_start, b.department_id,
           SUM(CASE WHEN a.category = 'Revenue' THEN b.budget_amount_usd ELSE 0 END) * any_value(p.factor) AS budget_revenue,
           SUM(CASE WHEN a.category <> 'Revenue' THEN b.budget_amount_usd ELSE 0 END) * any_value(p.factor) AS budget_expense
    FROM gold.fact_budget b
    JOIN gold.dim_gl_account a USING (account_id)
    CROSS JOIN LATERAL (SELECT CASE WHEN b.month_start = date_trunc('month', DATE '{AS_OF}')
                                    THEN day(DATE '{AS_OF}') / day(last_day(DATE '{AS_OF}')) ELSE 1 END AS factor) p
    WHERE b.is_latest_version
    GROUP BY 1, 2
)
SELECT act.month_start,
       CAST(strftime(act.month_start, '%Y%m') AS INTEGER) AS month_sk,
       act.department_id, dep.dept_code, dep.department_name,
       act.revenue_actual, act.expense_actual,
       coalesce(bud.budget_expense, 0) AS budget_expense,
       ROUND(100.0 * (act.revenue_actual - act.expense_actual) / NULLIF(act.revenue_actual, 0), 2) AS net_margin_pct,
       ROUND(100.0 * (act.expense_actual - bud.budget_expense) / NULLIF(bud.budget_expense, 0), 2) AS expense_variance_pct,
       ROUND(act.expense_actual / NULLIF(act.revenue_actual, 0), 4) AS burn_rate,
       coalesce(bud.budget_revenue, 0) AS budget_revenue,
       ROUND(100.0 * (act.revenue_actual - bud.budget_revenue) / NULLIF(bud.budget_revenue, 0), 2) AS revenue_variance_pct,
       act.month_start = date_trunc('month', DATE '{AS_OF}') AS is_month_to_date
FROM actuals act
JOIN gold.dim_department dep USING (department_id)
LEFT JOIN budgets bud USING (month_start, department_id)
""")
con.execute("""
CREATE OR REPLACE TABLE gold.monthly_financials AS
SELECT month_start,
       CAST(strftime(month_start, '%Y%m') AS INTEGER) AS month_sk,
       SUM(revenue_actual) AS total_revenue,
       SUM(expense_actual) AS total_expense,
       SUM(revenue_actual) - SUM(expense_actual) AS net_profit,
       ROUND(100.0 * (SUM(revenue_actual) - SUM(expense_actual)) / NULLIF(SUM(revenue_actual), 0), 2) AS net_margin_pct,
       ROUND(100.0 * (SUM(expense_actual) / NULLIF(SUM(budget_expense), 0) - 1), 2) AS expense_vs_budget_pct,
       SUM(budget_expense) AS total_budget_expense,
       SUM(budget_revenue) AS total_budget_revenue
FROM gold.kpi_monthly_actual_vs_budget
GROUP BY month_start
""")
con.execute(f"""
CREATE OR REPLACE TABLE gold.kpi_ap_aging AS
SELECT DATE '{AS_OF}' AS as_of_date, f.aging_bucket, v.vendor_category,
       count(*) AS invoice_count, SUM(f.open_amount_usd) AS open_amount_usd
FROM gold.fact_ap_invoices f JOIN gold.dim_vendor v USING (vendor_id)
WHERE f.aging_bucket <> 'Closed'
GROUP BY ALL
""")
con.execute("""
CREATE OR REPLACE TABLE gold.kpi_revenue_by_segment AS
SELECT dd.month_start, c.segment, c.region,
       SUM(f.amount_usd) AS revenue_usd,
       count(DISTINCT f.customer_id) AS active_customers,
       count(*) AS transactions
FROM gold.fact_transactions f
JOIN gold.dim_date dd USING (date_sk)
JOIN gold.dim_gl_account a USING (account_id)
JOIN gold.dim_customer c USING (customer_id)
WHERE a.category = 'Revenue'
GROUP BY ALL
""")
show("SELECT * FROM gold.monthly_financials ORDER BY month_start DESC LIMIT 6")

month_start,month_sk,total_revenue,total_expense,net_profit,net_margin_pct,expense_vs_budget_pct,total_budget_expense,total_budget_revenue
2026-10-01,202610,254261776.92,211430745.04,42831031.88,16.85,-3.47,2.190406e+08,2.797132e+08
2026-09-01,202609,340944596.35,275547934.91,65396661.44,19.18,-2.57,2.828196e+08,3.640823e+08
2026-08-01,202608,251295347.16,206820696.92,44474650.24,17.70,-0.45,2.077508e+08,2.717015e+08
2026-07-01,202607,238467267.62,194108554.86,44358712.76,18.60,-1.39,1.968489e+08,2.548809e+08
2026-06-01,202606,304255554.55,250905041.94,53350512.61,17.53,-0.77,2.528639e+08,3.289591e+08
2026-05-01,202605,275169052.84,219821665.80,55347387.04,20.11,-4.81,2.309173e+08,2.877842e+08


## 4 · Production tests
Primary keys unique, every foreign key resolves, measures not null, gold reconciles to silver to the cent, one
latest budget per key, no future-dated facts, data fresh. `error` failures stop the pipeline; `warn` is logged only.

In [12]:
FACT_FKS = {
    "fact_transactions": [("date_sk", "dim_date", "date_sk"), ("department_id", "dim_department", "department_id"),
                          ("account_id", "dim_gl_account", "account_id"), ("vendor_id", "dim_vendor", "vendor_id"),
                          ("customer_id", "dim_customer", "customer_id")],
    "fact_budget": [("date_sk", "dim_date", "date_sk"), ("department_id", "dim_department", "department_id"),
                    ("account_id", "dim_gl_account", "account_id")],
    "fact_ap_invoices": [("vendor_id", "dim_vendor", "vendor_id"), ("invoice_date_sk", "dim_date", "date_sk"),
                         ("due_date_sk", "dim_date", "date_sk")],
}
PRIMARY_KEYS = {"dim_date": "date_sk", "dim_department": "department_id", "dim_gl_account": "account_id",
                "dim_vendor": "vendor_id", "dim_customer": "customer_id", "fact_transactions": "transaction_sk",
                "fact_budget": "budget_sk", "fact_ap_invoices": "invoice_sk",
                "dim_department_history": "department_version_id"}

tests: list[tuple[str, str, str, str]] = []  # (table, test, severity, sql returning failing-row count)
for table, pk in PRIMARY_KEYS.items():
    tests.append((table, f"pk_unique_{pk}", "error", f"SELECT count(*) - count(DISTINCT {pk}) FROM gold.{table}"))
    tests.append((table, f"pk_not_null_{pk}", "error", f"SELECT count(*) FROM gold.{table} WHERE {pk} IS NULL"))
for table, fks in FACT_FKS.items():
    for col, dim, dim_col in fks:
        tests.append((table, f"fk_{col}_in_{dim}", "error",
                      f"SELECT count(*) FROM gold.{table} f LEFT JOIN gold.{dim} d ON f.{col} = d.{dim_col} WHERE d.{dim_col} IS NULL"))
tests += [
    ("fact_transactions", "amount_usd_not_null", "error", "SELECT count(*) FROM gold.fact_transactions WHERE amount_usd IS NULL"),
    ("fact_transactions", "reconciles_to_silver_rows", "error",
     "SELECT abs((SELECT count(*) FROM silver.silver_transactions WHERE NOT is_test_customer) - (SELECT count(*) FROM gold.fact_transactions))"),
    ("fact_transactions", "reconciles_to_silver_amount_cents", "error",
     "SELECT round(abs((SELECT sum(amount_usd) FROM silver.silver_transactions WHERE NOT is_test_customer) "
     "- (SELECT sum(amount_usd) FROM gold.fact_transactions)) * 100)"),
    ("fact_transactions", "no_future_dated_rows", "error",
     f"SELECT count(*) FROM gold.fact_transactions f JOIN gold.dim_date d USING (date_sk) WHERE d.full_date > DATE '{AS_OF}' + 1"),
    ("fact_transactions", "freshness_last_3_days", "warn",
     f"SELECT CASE WHEN max(d.full_date) >= DATE '{AS_OF}' - 3 THEN 0 ELSE 1 END FROM gold.fact_transactions f JOIN gold.dim_date d USING (date_sk)"),
    ("fact_transactions", "unknown_department_share_below_1pct", "warn",
     "SELECT CASE WHEN avg((department_id = -1)::INT) < 0.01 THEN 0 ELSE count(*) FILTER (WHERE department_id = -1) END FROM gold.fact_transactions"),
    ("kpi_monthly_actual_vs_budget", "reconciles_to_fact", "error",
     "SELECT round(abs((SELECT sum(revenue_actual + expense_actual) FROM gold.kpi_monthly_actual_vs_budget) "
     "- (SELECT sum(amount_usd) FROM gold.fact_transactions)) * 100)"),
    ("fact_budget", "one_latest_version_per_key", "error",
     "SELECT count(*) FROM (SELECT month_start, department_id, account_id FROM gold.fact_budget WHERE is_latest_version "
     "AND department_id > 0 GROUP BY ALL HAVING count(*) > 1)"),
    ("fact_ap_invoices", "open_amount_not_negative_for_invoices", "warn",
     "SELECT count(*) FROM gold.fact_ap_invoices WHERE doc_type = 'INVOICE' AND open_amount_usd < 0"),
    ("dim_vendor", "unknown_and_na_members_present", "error",
     "SELECT 2 - count(*) FROM gold.dim_vendor WHERE vendor_id IN (-1, -2)"),
]

results = []
for table, test, severity, sql in tests:
    failed = int(scalar(sql) or 0)
    total = int(scalar(f"SELECT count(*) FROM gold.{table}"))
    results.append((table, test, severity, failed))
    con.execute("INSERT INTO audit.dq_results VALUES (?, 'gold', ?, ?, ?, ?, ?, ?, ?, now())",
                [RUN_ID, table, test, severity, failed, total,
                 round(100.0 * (1 - failed / total), 3) if total else 100.0, "tested"])

import pandas as pd  # noqa: E402  (only for the summary table)

summary = pd.DataFrame(results, columns=["table", "test", "severity", "failed"])
display(summary[summary["failed"] > 0] if (summary["failed"] > 0).any() else summary.groupby("severity").size().rename("passed").reset_index())
errors = summary[(summary["severity"] == "error") & (summary["failed"] > 0)]
print(f"{len(summary)} tests run, {len(errors)} errors, {int(((summary['severity'] == 'warn') & (summary['failed'] > 0)).sum())} warnings")

39 tests run, 0 errors, 0 warnings


severity,passed
error,36
warn,3


In [13]:
show("""
SELECT table_name, estimated_size AS rows, column_count
FROM duckdb_tables() WHERE schema_name = 'gold' ORDER BY estimated_size DESC
""")
print(f"silver -> gold finished in {time.time() - _t0:.1f}s")
con.execute("CHECKPOINT")
con.close()
if len(errors):
    raise AssertionError("gold tests failed: " + ", ".join(f"{r.table}.{r.test} ({r.failed})" for r in errors.itertuples()))

silver -> gold finished in 12.2s


table_name,rows,column_count
fact_transactions,1993263,17
fact_ap_invoices,148482,17
fact_budget,53686,9
dim_customer,50022,9
dim_vendor,5407,8
dim_date,1096,16
kpi_monthly_actual_vs_budget,1025,14
kpi_revenue_by_segment,420,6
dim_department_history,89,11
dim_gl_account,61,7
